# Analisis Biwenger

Notebook interactivo para explorar el mercado, las plantillas de tus rivales y la tuya, con ratios puntos/valor de mercado y puntos/clausula (temporada actual y anterior), rankings de rendimiento, valor justo de mercado, sugerencia de fichajes/ventas, calendario/forma, ofertas, e historial.

Toda la logica de carga/analisis vive en `biwenger_helpers.py` (compartida con `02_generar_excel.py`) para no duplicar codigo; este notebook es sobre todo para explorar los datos a mano.

**Antes de nada**: copia `.env.example` a `.env` y rellena tus credenciales si no lo has hecho ya.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

import biwenger_helpers
from biwenger_client import BiwengerClient

load_dotenv()

EMAIL = os.getenv("BIWENGER_EMAIL")
PASSWORD = os.getenv("BIWENGER_PASSWORD")
LEAGUE_ID = os.getenv("BIWENGER_LEAGUE_ID") or None
OWN_TEAM_ID = os.getenv("BIWENGER_OWN_TEAM_ID") or None

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Cargar datos

La primera vez descarga todo de la API (mercado, las plantillas de todos los managers, y el detalle de cada jugador implicado) y lo guarda en `output/cache/*.json`. Las siguientes veces reusa esa cache al instante.

**Es resumible y se detiene sola en el limite**: con `parar_en_primer_limite=True` (activado por defecto en esta celda), en cuanto la API corte por exceso de peticiones (~200 seguidas), la celda para ahi mismo -- sin quedarse en pausas largas -- y deja guardado en cache lo conseguido hasta ese momento (que se combina con lo de ejecuciones anteriores). Si `data` sale incompleta, `construir_dataframe` de la siguiente celda simplemente muestra menos jugadores de los esperados, no falla. Vuelve a ejecutar esta celda las veces que haga falta (en distintos momentos si quieres) para ir completando la cache poco a poco hasta que no falte nadie.

**Se refresca sola cada 24h**: si la cache esta completa pero tiene mas de `max_antiguedad_horas` (24 por defecto), esta celda la refresca automaticamente con datos de jugador frescos de verdad -- no hace falta acordarse de pasar `forzar_refresco=True` a mano cada dia. Pasa `max_antiguedad_horas=None` si prefieres desactivar esto y controlar el refresco tu mismo. Una cache incompleta se retoma siempre, sin importar la antiguedad.

Y como siempre, `forzar_refresco=True` fuerza un refresco completo ahora mismo, pase lo que pase con la antiguedad.

In [ ]:
data = biwenger_helpers.cargar_datos_liga(
    EMAIL, PASSWORD, league_id=LEAGUE_ID, own_team_id=OWN_TEAM_ID,
    forzar_refresco=False, parar_en_primer_limite=True,
)

print(f"score_id de tu liga: {data['score_id']}")
print(f"tu team_id: {data['mi_team_id']}")
print(f"tu saldo actual: {data['balance']:,} €")

In [ ]:
df = biwenger_helpers.construir_dataframe(data)
print(f"{len(df)} jugadores en total ({df['Origen'].nunique()} origenes distintos)")
df.head(20)

In [ ]:
# Exportar el DataFrame completo a Excel si prefieres revisarlo fuera del notebook
# (para un Excel ya formateado con 3 hojas -- Mercado/Rivales/Mi equipo -- usa 02_generar_excel.py)
# df.to_excel("output/mi_analisis.xlsx", index=False)

## 2. Explorar

`df` tiene una fila por jugador (mercado + cada rival + tu equipo), con la columna `Origen` para filtrar. Columnas principales:

- `Puntos temporada actual` / `anterior`, `Partidos temporada actual` / `anterior`: totales de la temporada en curso y de la anterior de LaLiga (si jugo en primera). `Temporada actual` / `anterior` llevan la etiqueta real (p.ej. "Temporada 2025/2026") por si hace falta consultarla -- el nombre de columna es siempre el mismo, no cambia segun el jugador. Se calculan por MAYORIA entre todos los jugadores (`temporada_liga_mas_comun`): si un jugador no ha jugado ni esta temporada ni la anterior, salen NaN en vez de arrastrar puntos de hace 2+ temporadas como si fueran de ahora.
- `Ratio pts/VM (actual)` / `(anterior)`: esos puntos totales entre valor de mercado (en millones).
- `Ratio pts totales/clausula` / `Ratio pts medios/clausula` (`actual`/`anterior`): dos formas de ver el mismo ratio contra el precio de la clausula -- por puntos TOTALES acumulados, o por puntos MEDIOS (por partido). Solo para jugadores de rivales/tuyos, no de mercado. Un jugador con pocos partidos jugados (lesion, fichaje reciente) puede salir mal en el total pero bien en la media, y viceversa -- mira las dos.
- `Forma (ult. partidos)`: media ponderada de los ultimos partidos, dando mas peso a los mas recientes.
- `Puntuacion potencial`: mezcla `Forma` (70%) con los puntos/partido de la temporada anterior (30%); es la metrica que usan los rankings/recomendadores de mas abajo.
- `Disponible` / `Estado`: False si esta lesionado/sancionado/descartado (ver `ESTADOS_NO_DISPONIBLE` en `biwenger_helpers.py`; los recomendadores los excluyen por defecto). `'doubt'` (duda) no cuenta como no disponible -- se ve en `Estado` pero no se filtra, porque suele significar que si podria jugar.
- `Proximo rival`, `Local/Visitante`, `Dificultad proximo partido`: su siguiente partido de LaLiga.
- `Tendencia precio (7d) %`: variacion de su valor de mercado en los ultimos 7 dias.

**`Origen` == `'Mercado'` significa jugador LIBRE de verdad** (sin dueno, comprable directo): las entradas del mercado que SI tienen dueno (alguien de tu liga puso a su jugador en venta) no cuentan como 'Mercado' -- la unica via real de ficharlos es el clausulazo, asi que solo aparecen como `'Rival: <nombre>'` con su `Clausula`. Ademas, `mercado/rivales` ya excluyen "morralla": jugadores sin equipo de 1a, con valor <= 400k, o inactivos (sin debutar esta temporada y menos de la mitad de partidos la anterior). Tu propio equipo nunca se filtra -- ahi ves todo lo que tienes, este activo o no.

Algunos ejemplos para arrancar:

In [ ]:
# Top 20 gangas del mercado por ratio puntos/valor de mercado
(
    df[df["Origen"] == "Mercado"]
    .dropna(subset=["Ratio pts/VM (actual)"])
    .sort_values("Ratio pts/VM (actual)", ascending=False)
    .head(20)
)

In [ ]:
# Tu plantilla ordenada por Puntuacion potencial (para ver tus jugadores mas flojos)
df[df["Origen"] == "Mi equipo"].sort_values("Puntuacion potencial")

In [ ]:
# Mejores clausulas de rivales por ratio pts totales/clausula (cambia a "medios" para ver por puntos por partido)
(
    df[df["Origen"].str.startswith("Rival:")]  # añade "& (df["Posicion"] == "Delantero")" para filtrar por posicion
    .dropna(subset=["Ratio pts totales/clausula (actual)"])
    .sort_values("Ratio pts totales/clausula (actual)", ascending=False)
    .head(20)
)

In [ ]:
# Busca un jugador concreto por nombre exacto
df[df["Jugador"] == "Nombre Apellido"]

## 3. Graficos

In [ ]:
top20 = df.dropna(subset=["Ratio pts/VM (actual)"]).nlargest(20, "Ratio pts/VM (actual)")

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top20["Jugador"], top20["Ratio pts/VM (actual)"])
ax.invert_yaxis()
ax.set_xlabel("Ratio pts/VM (actual)")
ax.set_title("Top 20 jugadores por ratio puntos / valor de mercado")
plt.tight_layout()
plt.show()

In [ ]:
posiciones = ["Portero", "Defensa", "Centrocampista", "Delantero"]
datos = [df[df["Posicion"] == p]["Ratio pts/VM (actual)"].dropna() for p in posiciones]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(datos, tick_labels=posiciones)
ax.set_ylabel("Ratio pts/VM (actual)")
ax.set_title("Distribucion del ratio pts/VM por posicion")
plt.tight_layout()
plt.show()

### Valor vs rendimiento -- chollos y sobrevalorados

Cada punto es un jugador: eje X el valor de mercado, eje Y su `Puntuacion potencial`. La linea discontinua es la tendencia general (a mas precio, mas rendimiento esperado). Los que quedan muy por ENCIMA de la linea rinden mas de lo que su precio sugiere (chollos, en verde); los que quedan muy por DEBAJO rinden menos de lo esperado para su precio (sobrevalorados, en rojo) -- los 8 outliers de cada lado salen rotulados con su nombre.

In [ ]:
plot_df = biwenger_helpers.unico_por_jugador(df.dropna(subset=["Valor de mercado", "Puntuacion potencial"]))
plot_df = plot_df[plot_df["Valor de mercado"] > 0].copy()
plot_df["VM (M€)"] = plot_df["Valor de mercado"] / 1_000_000

COLOR_POSICION = {
    "Portero": "#1f77b4", "Defensa": "#2ca02c",
    "Centrocampista": "#ff7f0e", "Delantero": "#d62728",
}

fig, ax = plt.subplots(figsize=(11, 7))
for pos, color in COLOR_POSICION.items():
    sub = plot_df[plot_df["Posicion"] == pos]
    ax.scatter(sub["VM (M€)"], sub["Puntuacion potencial"], label=pos, alpha=0.45, color=color, s=22)

# tendencia general (regresion lineal simple) para detectar quien se desvia mas
x = plot_df["VM (M€)"].to_numpy()
y = plot_df["Puntuacion potencial"].to_numpy()
pendiente, intercepto = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, pendiente * xs + intercepto, "k--", alpha=0.5, linewidth=1, label="tendencia general")

plot_df["_residuo"] = y - (pendiente * x + intercepto)
N_OUTLIERS = 8
chollos = plot_df.nlargest(N_OUTLIERS, "_residuo")
sobrevalorados = plot_df.nsmallest(N_OUTLIERS, "_residuo")

for _, row in chollos.iterrows():
    ax.annotate(row["Jugador"], (row["VM (M€)"], row["Puntuacion potencial"]),
                fontsize=8, color="#1a7a1a", fontweight="bold",
                xytext=(5, 4), textcoords="offset points")
for _, row in sobrevalorados.iterrows():
    ax.annotate(row["Jugador"], (row["VM (M€)"], row["Puntuacion potencial"]),
                fontsize=8, color="#b31212", fontweight="bold",
                xytext=(5, -9), textcoords="offset points")

ax.scatter(chollos["VM (M€)"], chollos["Puntuacion potencial"], facecolors="none", edgecolors="#1a7a1a", s=90, linewidths=1.5)
ax.scatter(sobrevalorados["VM (M€)"], sobrevalorados["Puntuacion potencial"], facecolors="none", edgecolors="#b31212", s=90, linewidths=1.5)

ax.set_xlabel("Valor de mercado (M€)")
ax.set_ylabel("Puntuacion potencial")
ax.set_title("Valor vs rendimiento -- outliers rotulados (verde = chollo, rojo = sobrevalorado)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
plot_df.sort_values('_residuo', ascending=False).head(20)

In [ ]:
plot_df.sort_values('Ratio pts medios/clausula (actual)', ascending=False).head(20)

### Mejores y peores por posicion

Los 5 con mejor y los 5 con peor `Puntuacion potencial` de cada posicion, entre TODOS los jugadores (mercado + rivales + tu equipo -- el origen sale entre parentesis). Util para ver de un vistazo quien despunta y quien va mal en cada linea del equipo.

In [ ]:
N_POR_LADO = 5
posiciones = ["Portero", "Defensa", "Centrocampista", "Delantero"]
df_unico = biwenger_helpers.unico_por_jugador(df)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, pos in zip(axes.flat, posiciones):
    sub = df_unico[df_unico["Posicion"] == pos].dropna(subset=["Puntuacion potencial"])
    peores = sub.nsmallest(N_POR_LADO, "Puntuacion potencial").sort_values("Puntuacion potencial", ascending=False)
    mejores = sub.nlargest(N_POR_LADO, "Puntuacion potencial").sort_values("Puntuacion potencial", ascending=True)
    combo = pd.concat([peores, mejores])
    etiquetas = combo["Jugador"] + " (" + combo["Origen"].str.slice(0, 14) + ")"
    colores = ["#b31212"] * len(peores) + ["#1a7a1a"] * len(mejores)

    ax.barh(etiquetas, combo["Puntuacion potencial"], color=colores)
    ax.set_title(pos)
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Mejores (verde) y peores (rojo) por posicion -- Puntuacion potencial", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Sugerencia de fichajes

Compara, para cada candidato asequible y disponible (clausula de un rival o precio de mercado, dentro de tu saldo actual), la `Puntuacion potencial` contra tu jugador mas flojo de esa misma posicion. Solo aparecen candidatos que mejorarian a ese jugador. Las clausulas de rival que esten bloqueadas ahora mismo (`Clausula disponible ahora` == False, tras una compra/clausulazo reciente -- la API la rechazaria) se descartan siempre, aunque el resto te encaje.

Columnas utiles para decidir:
- `Saldo tras fichar`: lo que te quedaria de saldo despues de este fichaje.
- `Jugadores flojos en tu posicion` / `Total en tu posicion`: cuantos de tus jugadores en esa posicion rinden por debajo de tu propia mediana ahi -- mas de uno indica una posicion con mas de un eslabon debil, no solo el peor.
- `Mejora por millon gastado`: la mejora normalizada por coste. Usa `ordenar_por="eficiencia"` para priorizar fichajes baratos que mejoran poco a poco frente a uno carisimo que mejora mucho de golpe.

Esto es una sugerencia orientativa, no tiene en cuenta cosas como si ya tienes hueco en la plantilla o el calendario de proximos partidos (para eso mira la columna `Dificultad proximo partido` del `df` en la seccion 2).

In [ ]:
recomendaciones = biwenger_helpers.sugerir_fichajes(df, data["balance"], top_n=15)
recomendaciones

In [ ]:
# Mismo calculo, pero priorizando fichajes eficientes (mejora por millon) en vez de mejora absoluta
biwenger_helpers.sugerir_fichajes(df, data["balance"], top_n=15, ordenar_por="eficiencia")

## 5. Ranking general de jugadores

`mejores_jugadores` es una watchlist: quien esta rindiendo mejor ahora mismo (por `Puntuacion potencial`), sin mirar precio ni si te lo puedes permitir. Filtra por posicion y/o por origen (`'Mercado'`, `'Mi equipo'`, o `'Rival: <nombre>'`). Excluye lesionados/sancionados por defecto.

In [ ]:
# Top 20 general, todas las posiciones y origenes
biwenger_helpers.mejores_jugadores(df, top_n=20)

In [ ]:
# Ejemplo filtrado: mejores delanteros que esten en el mercado
biwenger_helpers.mejores_jugadores(df, posicion="Delantero", origen="Mercado", top_n=15)

In [ ]:
# El mejor portero entre todos los que tenemos datos (mercado + rivales + tu equipo).
# Si esta en 'Mercado' es libre (comprable directo); si es 'Rival: X' hace falta clausulazo.
mejor_portero = biwenger_helpers.mejores_jugadores(df, posicion="Portero", top_n=1)
mejor_portero

In [ ]:
# Top 10 porteros, para comparar alternativas (no solo el numero 1)
biwenger_helpers.mejores_jugadores(df, posicion="Portero", top_n=10)

## 6. Valor justo de mercado (chollos / sobreprecios)

Para cada jugador EN VENTA en el mercado, estima un "valor justo" a partir de como el resto de jugadores de tu liga (de su misma posicion) relacionan rendimiento (`Puntuacion potencial`) con valor de mercado -- usa la MEDIANA de esa relacion como referencia. Compara ese valor justo con el precio pedido para detectar chollos/sobreprecios y sugerir una oferta.

Es una heuristica basada en tu propia liga, no el precio "real" de Biwenger -- tratalo como orientacion. `margen_pct` controla a partir de que % de diferencia se etiqueta como Chollo/Caro (por defecto 20%).

In [ ]:
valor_justo = biwenger_helpers.estimar_valor_justo(df, margen_pct=20)
valor_justo

In [ ]:
# Solo los chollos, ordenados de mejor a peor
valor_justo[valor_justo["Valoracion"] == "Chollo"]

### Cuanto ofertar (solo mercado -- las clausulas son un importe fijo, no se puja)

`sugerir_oferta_mercado` combina tres cosas para cada jugador libre:
- Tu **valor justo estimado** (igual que arriba): nunca sugiere pujar por encima de esto.
- **Competencia real**: cuantos rivales tienen `maximumBid` (limite de puja de standings, puede ser mayor que su saldo actual) por encima del precio pedido. Si hay competencia, sugiere un margen sobre el precio pedido para no perderlo por poco.
- **Comparables reales**: la mediana de lo que los duenos actuales de jugadores similares (misma posicion, valor parecido) pagaron de verdad por ellos.

Es orientativo -- no sabemos quien esta REALMENTE interesado en cada jugador, solo quien podria permitirselo.

In [ ]:
biwenger_helpers.sugerir_oferta_mercado(df, data)

## 7. Sugerencia de ventas

El complementario a la seccion 6, pero para TU plantilla: compara el valor de mercado OFICIAL de cada jugador tuyo con lo que su rendimiento actual justificaria (misma mediana de referencia por posicion). Si esta muy por encima, esta sobrevalorado -- venderlo ahora aprovecha un precio que probablemente no se sostenga si no mejora su rendimiento.

Tambien cruza las **ofertas reales** que has recibido (seccion 8): si alguien ya te ofrece igual o mas que el valor justo estimado, se recomienda vender aunque el % de sobrevaloracion por si solo no llegase al umbral -- una oferta en la mano pesa mas que la heuristica.

In [ ]:
ventas = biwenger_helpers.sugerir_ventas(df, data=data, margen_pct=20)
ventas

In [ ]:
# Solo los que convendria vender ya
ventas[ventas["Recomendacion"] == "Vender ahora"]

## 8. Ofertas

Ofertas de compra activas en el mercado: `'recibidas'` son las que otros managers te han hecho por tus jugadores, `'enviadas'` las que tu has hecho. Esta libreria no incluye aceptar/rechazar -- eso hazlo desde la app; esto es solo para verlas listadas sin tener que entrar.

Nota: Biwenger no siempre identifica al postor en esta respuesta, por lo que `De`/`A` pueden salir como `'Desconocido'`.

In [ ]:
biwenger_helpers.ofertas(data, tipo="recibidas")

In [ ]:
biwenger_helpers.ofertas(data, tipo="enviadas")

## 9. Historial

A diferencia de la cache de la seccion 1 (que se sobrescribe cada refresco), `guardar_snapshot` guarda una copia fechada en `output/history/<timestamp>/`, para poder comparar la evolucion de tu equipo/el mercado con el tiempo. Usalo justo despues de un `forzar_refresco=True`.

In [ ]:
# Descomenta para guardar un snapshot fechado de los datos actuales
# biwenger_helpers.guardar_snapshot(data)

biwenger_helpers.listar_snapshots()

In [ ]:
# Ejemplo: comparar tu plantilla actual contra un snapshot anterior.
# Cambia el nombre por uno real de listar_snapshots().
# anterior_df = biwenger_helpers.construir_dataframe(biwenger_helpers.cargar_snapshot("20260828_120000"))
# comparacion = df.merge(
#     anterior_df[["player_id", "Puntuacion potencial", "Valor de mercado"]],
#     on="player_id", suffixes=("", " (antes)"),
# )
# comparacion["Cambio potencial"] = comparacion["Puntuacion potencial"] - comparacion["Puntuacion potencial (antes)"]
# comparacion[comparacion["Origen"] == "Mi equipo"].sort_values("Cambio potencial")

## 10. Operaciones (pujar / clausular) -- acciones REALES sobre tu liga

`place_offer` funciona en modo **dry-run por defecto**: sin `confirm=True` no se envia nada, solo se imprime el payload que se mandaria. Repasa siempre el dry-run antes de confirmar -- esto gasta/compromete tu saldo de verdad y **no se puede deshacer** desde aqui.

Necesita su propio login (no reusa la sesion de `cargar_datos_liga`, que es solo para lectura).

In [ ]:
cliente_operaciones = BiwengerClient(EMAIL, PASSWORD, league_id=LEAGUE_ID)
biwenger_helpers.login_y_resolver_liga(cliente_operaciones, LEAGUE_ID)
print("Sesion lista. Saldo (dato ya cargado en el paso 1):", data["balance"])

In [ ]:
# Ejemplo: puja/clausula por un jugador de 'recomendaciones', 'valor_justo' o el que quieras.
# 'Vendedor (id)' de esas tablas es lo que va en 'to'.
player_id = None  # <- pon aqui el player_id
importe = None  # <- importe de tu oferta (o el precio de la clausula; en valor_justo, 'Oferta sugerida')
vendedor_id = None  # <- 'Vendedor (id)' de la fila que te interese

if player_id and importe and vendedor_id:
    cliente_operaciones.place_offer(player_id, importe, to=vendedor_id)
    # Cuando el payload de arriba te parezca correcto, descomenta esta linea
    # para ejecutarla de verdad (gasta/compromete tu saldo):
    # cliente_operaciones.place_offer(player_id, importe, to=vendedor_id, confirm=True)
else:
    print("Rellena player_id / importe / vendedor_id antes de ejecutar esta celda.")